# 04B2 — Optuna Bayesian/Pareto challengers

Objectif : générer des modèles challengers avec Optuna, puis les comparer au panel robuste issu de 04B.

Protocole :
- calibration SIMCA : objets purs peanut batches 1–2 ;
- validation : objets purs almond + peanut batch 3 ;
- aucune image mixture ;
- aucun batch 4 ;
- études Optuna séparées pour `object_matrix` et `pixel_matrix` ;
- Optuna ne remplace pas la grille : il propose des challengers ;
- la sélection finale reste Pareto, sans score unique.

Différence avec 04B :
- 04B sélectionne un panel robuste depuis la grille exhaustive ;
- 04B2 cherche des configurations supplémentaires avec Optuna ;
- les candidats Optuna sont refittés, calibrés en 3-way sur validation, puis fusionnés avec le panel 04B pour le test externe 04C.

Sorties principales :
- `optuna_trials_object_matrix.parquet`
- `optuna_trials_pixel_matrix.parquet`
- `optuna_pareto_candidates.parquet`
- `optuna_validation_refit_metrics.parquet`
- `optuna_validation_3way_metrics.parquet`
- `optuna_challengers_for_04C.parquet`
- `candidate_configs_for_pure_test.parquet`

In [12]:
from __future__ import annotations

import sys
import json
import gc
import optuna
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 240)
pd.set_option("display.max_rows", 300)

CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src.io.database_h5 import load_nir_uco_h5

from src.utils import (
    save_parquet,
    save_parquet_if_nonempty,
    load_parquet,
    list_result_files,
)

from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)

from src.spectra.preprocessing_configs import normalize_preprocessing_configs

from src.decision.metrics import (
    add_binary_confusion_case,
)

from src.decision.uncertainty import (
    evaluate_three_way_by_config,
    calibrate_three_way_thresholds_by_config,
)

from src.workflows.simca import (
    make_target_train_filters,
    refit_selected_simca_configs,
)

from src.workflows.simca_selection_utils import (
    ensure_candidate_columns,
    normalize_simca_rule_columns,
    add_detection_selection_score,
    add_reference_selection_scores,
    fill_selected_config_defaults,
    summarize_metric_stability,
    pareto_front_by_group,
    materialize_selection_metrics,
)

from src.workflows.simca_optuna import (
    make_optuna_binary_pareto_objective,
    optuna_trials_dataframe,
    close_optuna_study,
)

%load_ext autoreload
%autoreload 2


PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

# ---------------------------------------------------------------------
# Spectral configuration
# ---------------------------------------------------------------------
USE_WAVELENGTH_WINDOW = False
WAVELENGTH_MODE = "non_noisy_all"

WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

if USE_WAVELENGTH_WINDOW:
    RESULTS_TAG = f"{int(WINDOW_MIN_NM)}_{int(WINDOW_MAX_NM)}"
else:
    RESULTS_TAG = "non_noisy_all"

# ---------------------------------------------------------------------
# Inputs
# ---------------------------------------------------------------------
RESULTS_03_DIR = PROJECT_ROOT / "results" / f"03_pca_{RESULTS_TAG}"
PCA_SELECTED_PREPROCESSINGS_PATH = RESULTS_03_DIR / "pca_selected_preprocessings.parquet"

RESULTS_04A_DIR = PROJECT_ROOT / "results" / f"04A_simca_grid_validation_{RESULTS_TAG}"
GRID_SELECTED_CANDIDATES_PATH = RESULTS_04A_DIR / "selected_candidate_configs.parquet"

RESULTS_04B_DIR = PROJECT_ROOT / "results" / f"04B_simca_validation_robustness_{RESULTS_TAG}"
ROBUST_CANDIDATE_CONFIGS_PATH = RESULTS_04B_DIR / "04B_final_candidate_panel.parquet"

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results" / f"04B2_optuna_challenge_{RESULTS_TAG}"
STUDY_DIR = RESULTS_DIR / "optuna_studies"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
STUDY_DIR.mkdir(parents=True, exist_ok=True)

OPTUNA_STORAGE_OBJECT_PATH = STUDY_DIR / f"simca_optuna_object_matrix_{RESULTS_TAG}.db"
OPTUNA_STORAGE_PIXEL_PATH = STUDY_DIR / f"simca_optuna_pixel_matrix_{RESULTS_TAG}.db"

OPTUNA_TRIALS_OBJECT_PATH = RESULTS_DIR / "optuna_trials_object_matrix.parquet"
OPTUNA_TRIALS_PIXEL_PATH = RESULTS_DIR / "optuna_trials_pixel_matrix.parquet"

OPTUNA_COMPLETED_TRIALS_OBJECT_PATH = RESULTS_DIR / "optuna_completed_trials_object_matrix.parquet"
OPTUNA_COMPLETED_TRIALS_PIXEL_PATH = RESULTS_DIR / "optuna_completed_trials_pixel_matrix.parquet"

OPTUNA_ALL_TRIALS_PATH = RESULTS_DIR / "optuna_trials_all_families.parquet"
OPTUNA_PARETO_CANDIDATES_PATH = RESULTS_DIR / "optuna_pareto_candidates.parquet"

OPTUNA_VALIDATION_REFIT_OBJECTS_PATH = RESULTS_DIR / "optuna_validation_refit_objects.parquet"
OPTUNA_VALIDATION_REFIT_PIXELS_PATH = RESULTS_DIR / "optuna_validation_refit_pixels.parquet"

OPTUNA_THREE_WAY_GRID_PATH = RESULTS_DIR / "optuna_three_way_threshold_grid.parquet"
OPTUNA_THREE_WAY_SELECTED_PATH = RESULTS_DIR / "optuna_three_way_selected_thresholds.parquet"
OPTUNA_VALIDATION_3WAY_OBJECTS_PATH = RESULTS_DIR / "optuna_validation_3way_objects.parquet"
OPTUNA_VALIDATION_3WAY_METRICS_PATH = RESULTS_DIR / "optuna_validation_3way_metrics.parquet"

OPTUNA_CHALLENGERS_FOR_04C_PATH = RESULTS_DIR / "optuna_challengers_for_04C.parquet"

OPTUNA_VALIDATION_REFIT_METRICS_PATH = RESULTS_DIR / "optuna_validation_refit_metrics.parquet"
OPTUNA_VALIDATION_PIXEL_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "optuna_validation_pixel_errors_by_image.parquet"
OPTUNA_VALIDATION_REFIT_ERRORS_PATH = RESULTS_DIR / "optuna_validation_refit_errors.parquet"

CANDIDATE_CONFIGS_FOR_PURE_TEST_PATH = RESULTS_DIR / "candidate_configs_for_pure_test.parquet"
OPTUNA_CHALLENGE_PROTOCOL_PATH = RESULTS_DIR / "optuna_challenge_protocol.parquet"

# ---------------------------------------------------------------------
# Protocol
# ---------------------------------------------------------------------
TARGET_CLASS = "peanut"
NON_TARGET_LABEL = "non_target"
REFERENCE_CLASSES = ("almond", TARGET_CLASS)

TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=[1, 2],
)

VALIDATION_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": list(REFERENCE_CLASSES),
    "batch": [3],
}

# ---------------------------------------------------------------------
# Optuna search spaces by matrix family
# ---------------------------------------------------------------------

OPTUNA_MATRIX_FAMILY_SPACES = {
    "object_matrix": {
        "matrix_methods": ["object_mean", "object_median"],
        "storage_path": OPTUNA_STORAGE_OBJECT_PATH,
        "trials_path": OPTUNA_TRIALS_OBJECT_PATH,
        "completed_trials_path": OPTUNA_COMPLETED_TRIALS_OBJECT_PATH,
        "study_name": f"simca_optuna_object_matrix_{RESULTS_TAG}",
    },
    "pixel_matrix": {
        "matrix_methods": ["balanced_pixels"],
        "storage_path": OPTUNA_STORAGE_PIXEL_PATH,
        "trials_path": OPTUNA_TRIALS_PIXEL_PATH,
        "completed_trials_path": OPTUNA_COMPLETED_TRIALS_PIXEL_PATH,
        "study_name": f"simca_optuna_pixel_matrix_{RESULTS_TAG}",
    },
}

OPTUNA_RULE_VARIANTS = [
    "simple_chi2",
    "simple_emp_cv",
    "alternative_chi2_fixed2",
    "alternative_chi2_emp_cv",
    "alternative_empHQ_fixed2",
    "alternative_empHQ_emp_cv",
    "data_driven_chi2",
    "data_driven_emp_cv",
    "combined_index_chi2",
]

OPTUNA_N_COMPONENTS_CHOICES = [3, 4, 5, 6, 7, 8, 10, 11, 12]
OPTUNA_ALPHA_CHOICES = [0.05, 0.01]

# Object threshold is calibrated inside the objective, not suggested by Optuna.
OPTUNA_OBJECT_THRESHOLDS = np.round(np.arange(0.50, 0.96, 0.05), 2)

OPTUNA_M_CHOICES = [20, 40, 60, 80]
OPTUNA_BALANCED_PIXEL_STRATEGY_CHOICES = ["random", "center"]

OPTUNA_SG_WINDOW_CHOICES = [7, 9, 11, 13, 15]
OPTUNA_SG_POLYORDER_CHOICES = [2]

OPTUNA_POSITION_DILATION_RADIUS_CHOICES = [2, 3, 4, 5]

# ---------------------------------------------------------------------
# Runtime
# ---------------------------------------------------------------------
RUN_OPTUNA = True
LOAD_EXISTING_STUDY = False

OPTUNA_N_TRIALS_PER_FAMILY = {
    "object_matrix": 200,
    "pixel_matrix": 300,
}

OPTUNA_OBJECTIVE_SEEDS = [0, 1, 2]
OPTUNA_MAX_FN_RATE_FOR_THRESHOLD = 0.00
OPTUNA_MAX_FP_RATE_FOR_THRESHOLD = 0.50

N_OPTUNA_PARETO_PER_FAMILY = 25
N_OPTUNA_REFIT_PER_FAMILY = 15

OPTUNA_TIMEOUT = None
OPTUNA_N_JOBS = 1

RANDOM_STATE = 42
REPLACE_BALANCED_PIXELS = False

CV_N_SPLITS = 5
CV_GROUP_COL = "object_id"

# Candidate selection from Optuna trials.
N_OPTUNA_CANDIDATES_PER_FAMILY = 8
N_OPTUNA_CANDIDATES_OVERALL = 20

# Merge with 04B robust grid candidates for 04C.
INCLUDE_GRID_ROBUST_CANDIDATES = True
N_GRID_CANDIDATES_TO_KEEP = 25

THREE_WAY_LOWER_THRESHOLDS = np.round(np.arange(0.05, 0.61, 0.05), 2)
THREE_WAY_UPPER_THRESHOLDS = np.round(np.arange(0.50, 0.96, 0.05), 2)
MAX_THREE_WAY_TARGET_MISS_RATE = 0.00
MAX_THREE_WAY_FALSE_ACCEPT_RATE = 0.30
MAX_THREE_WAY_UNCERTAIN_RATE = 0.60

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("TRAIN_FILTERS:", TRAIN_FILTERS)
print("VALIDATION_FILTERS:", VALIDATION_FILTERS)
print("PCA_SELECTED_PREPROCESSINGS_PATH:", PCA_SELECTED_PREPROCESSINGS_PATH)
print("ROBUST_CANDIDATE_CONFIGS_PATH:", ROBUST_CANDIDATE_CONFIGS_PATH)


DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B2_optuna_challenge_non_noisy_all
TRAIN_FILTERS: {'sample_kind': ['pure'], 'object_nut_type': ['peanut'], 'batch': [1, 2]}
VALIDATION_FILTERS: {'sample_kind': ['pure'], 'object_nut_type': ['almond', 'peanut'], 'batch': [3]}
PCA_SELECTED_PREPROCESSINGS_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_selected_preprocessings.parquet
ROBUST_CANDIDATE_CONFIGS_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_validation_robustness_non_noisy_all\04B_final_candidate_panel.parquet


In [3]:
if not DB_H5_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_H5_PATH}. Run notebook 00 first.")

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )
    wavelength_selection_df = wavelength_selection_summary(wavelength_info)
else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

wavelength_config_df = pd.DataFrame([{
    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_bands": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),
}])

print("Database loaded")
print("n objects:", len(object_db))
print("n images:", len(image_db))
display(wavelength_config_df)

if PCA_SELECTED_PREPROCESSINGS_PATH.exists():
    pca_selected_preprocessings_df = load_parquet(PCA_SELECTED_PREPROCESSINGS_PATH)
else:
    pca_selected_preprocessings_df = pd.DataFrame()

if GRID_SELECTED_CANDIDATES_PATH.exists():
    grid_selected_candidates_df = load_parquet(GRID_SELECTED_CANDIDATES_PATH)
else:
    grid_selected_candidates_df = pd.DataFrame()

if ROBUST_CANDIDATE_CONFIGS_PATH.exists():
    robust_grid_candidates_df = load_parquet(ROBUST_CANDIDATE_CONFIGS_PATH)
else:
    robust_grid_candidates_df = pd.DataFrame()

if len(robust_grid_candidates_df) == 0:
    raise FileNotFoundError(
        f"No robust 04B candidate panel found at: {ROBUST_CANDIDATE_CONFIGS_PATH}. "
        "Run 04B first."
    )

robust_grid_candidates_df = ensure_candidate_columns(robust_grid_candidates_df)
robust_grid_candidates_df = normalize_simca_rule_columns(robust_grid_candidates_df)
robust_grid_candidates_df = fill_selected_config_defaults(
    robust_grid_candidates_df,
    default_values={
        "target_class": TARGET_CLASS,
        "non_target_label": NON_TARGET_LABEL,
        "sg_window_length": 11,
        "sg_polyorder": 2,
        "position_dilation_radius": 3,
        "m": 40,
        "alpha": 0.05,
        "object_threshold": 0.75,
    },
)

print("PCA selected preprocessings:", pca_selected_preprocessings_df.shape)
print("04A grid candidates:", grid_selected_candidates_df.shape)
print("04B final candidate panel:", robust_grid_candidates_df.shape)
display(robust_grid_candidates_df["matrix_family"].value_counts(dropna=False))

Database loaded
n objects: 1262
n images: 48


,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


PCA selected preprocessings: (17, 11)
04A grid candidates: (22, 68)
04B final candidate panel: (18, 57)


matrix_family
object_matrix    12
pixel_matrix      6
Name: count, dtype: int64

In [4]:
required_04b_cols = [
    "selected_config_id",
    "matrix_family",
    "object_threshold",
    "three_way_lower_threshold",
    "three_way_upper_threshold",
]

missing_04b_cols = [
    col for col in required_04b_cols
    if col not in robust_grid_candidates_df.columns
]

if missing_04b_cols:
    raise KeyError(
        "04B final candidate panel is missing required columns: "
        f"{missing_04b_cols}. Re-run 04B after the 3-way/Pareto refactor."
    )

In [5]:
def _parse_preprocessing_steps(value):
    if isinstance(value, (list, tuple)):
        return tuple(str(v) for v in value)

    value = str(value)

    if "+" in value:
        return tuple(v.strip() for v in value.split("+") if v.strip())

    return (value.strip(),)


DEFAULT_OPTUNA_PREPROCESSING_CONFIGS = {
    "snv": ("snv",),
    "absorbance": ("absorbance",),
    "absorbance_snv": ("absorbance", "snv"),
    "absorbance_sg_smooth": ("absorbance", "sg_smooth"),
    "absorbance_sg_d1": ("absorbance", "sg_d1"),
    "absorbance_snv_sg_smooth": ("absorbance", "snv", "sg_smooth"),
    "absorbance_snv_sg_d1": ("absorbance", "snv", "sg_d1"),
    "snv_sg_d1": ("snv", "sg_d1"),
}

if len(robust_grid_candidates_df) > 0 and "preprocessing_steps" in robust_grid_candidates_df.columns:
    PREPROCESSING_CONFIGS = {
        str(row["preprocessing"]): _parse_preprocessing_steps(row["preprocessing_steps"])
        for _, row in robust_grid_candidates_df.drop_duplicates("preprocessing").iterrows()
    }
    print("Using preprocessing configs from 04B robust candidate panel.")

elif len(grid_selected_candidates_df) > 0 and "preprocessing_steps" in grid_selected_candidates_df.columns:
    PREPROCESSING_CONFIGS = {
        str(row["preprocessing"]): _parse_preprocessing_steps(row["preprocessing_steps"])
        for _, row in grid_selected_candidates_df.drop_duplicates("preprocessing").iterrows()
    }
    print("Using preprocessing configs from 04A selected candidates.")

elif len(pca_selected_preprocessings_df) > 0:
    PREPROCESSING_CONFIGS = {
        str(row["preprocessing"]): _parse_preprocessing_steps(row["preprocessing_steps"])
        for _, row in pca_selected_preprocessings_df.drop_duplicates("preprocessing").iterrows()
    }
    print("Using PCA preprocessing shortlist.")

else:
    PREPROCESSING_CONFIGS = DEFAULT_OPTUNA_PREPROCESSING_CONFIGS.copy()
    print("[WARNING] Using default Optuna preprocessing configs.")

PREPROCESSING_CONFIGS = normalize_preprocessing_configs(PREPROCESSING_CONFIGS)

preprocessing_config_df = pd.DataFrame([
    {
        "preprocessing": name,
        "preprocessing_steps": "+".join(steps),
        "uses_sg": any(str(step).startswith("sg_") for step in steps),
    }
    for name, steps in PREPROCESSING_CONFIGS.items()
])

print("Preprocessing configs used by Optuna:", len(PREPROCESSING_CONFIGS))
display(preprocessing_config_df)

Using preprocessing configs from 04B robust candidate panel.
Preprocessing configs used by Optuna: 7


,preprocessing,preprocessing_steps,uses_sg
0,absorbance_sg_smooth,absorbance+sg_smooth,True
1,absorbance_sg_d2,absorbance+sg_d2,True
2,raw,raw,False
3,sg_smooth,sg_smooth,True
4,snv_sg_smooth,snv+sg_smooth,True
5,absorbance_snv_sg_d1,absorbance+snv+sg_d1,True
6,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth,True


In [8]:
# ---------------------------------------------------------------------
# Run Optuna studies separately by matrix family
# ---------------------------------------------------------------------

optuna_trials_parts = []
optuna_completed_parts = []
studies = {}

for matrix_family, space in OPTUNA_MATRIX_FAMILY_SPACES.items():
    print("=" * 80)
    print(f"Optuna study for matrix_family={matrix_family}")
    print("=" * 80)

    storage_path = Path(space["storage_path"])
    storage_path.parent.mkdir(parents=True, exist_ok=True)

    objective = make_optuna_binary_pareto_objective(
        object_db=object_db,
        image_db=image_db,
        train_filters=TRAIN_FILTERS,
        projection_filters=VALIDATION_FILTERS,
        preprocessing_configs=PREPROCESSING_CONFIGS,
        matrix_methods=space["matrix_methods"],
        rule_variants=OPTUNA_RULE_VARIANTS,
        object_thresholds=OPTUNA_OBJECT_THRESHOLDS,
        seeds=OPTUNA_OBJECTIVE_SEEDS,
        n_components_choices=OPTUNA_N_COMPONENTS_CHOICES,
        alpha_choices=OPTUNA_ALPHA_CHOICES,
        m_choices=OPTUNA_M_CHOICES,
        balanced_pixel_strategy_choices=OPTUNA_BALANCED_PIXEL_STRATEGY_CHOICES,
        sg_window_choices=OPTUNA_SG_WINDOW_CHOICES,
        sg_polyorder_choices=OPTUNA_SG_POLYORDER_CHOICES,
        position_dilation_radius_choices=OPTUNA_POSITION_DILATION_RADIUS_CHOICES,
    )

    sampler = optuna.samplers.TPESampler(
        seed=RANDOM_STATE,
        multivariate=True,
        group=True,
        n_startup_trials=30,
    )

    storage = optuna.storages.RDBStorage(
        url=f"sqlite:///{storage_path.as_posix()}",
        engine_kwargs={"connect_args": {"timeout": 30.0}},
    )

    if RUN_OPTUNA:
        study = optuna.create_study(
            study_name=space["study_name"],
            directions=["minimize", "minimize", "maximize"],
            sampler=sampler,
            storage=storage,
            load_if_exists=LOAD_EXISTING_STUDY,
        )

        study.optimize(
            objective,
            n_trials=int(OPTUNA_N_TRIALS_PER_FAMILY[matrix_family]),
            timeout=OPTUNA_TIMEOUT,
            n_jobs=OPTUNA_N_JOBS,
            show_progress_bar=True,
        )

        close_optuna_study(study)

        trials_df = optuna_trials_dataframe(study)

    else:
        if not Path(space["trials_path"]).exists():
            raise FileNotFoundError(
                f"RUN_OPTUNA=False but missing trials file: {space['trials_path']}"
            )
        trials_df = load_parquet(space["trials_path"])

    trials_df = trials_df.copy()
    trials_df["matrix_family_study"] = matrix_family

    save_parquet(trials_df, space["trials_path"])

    completed_df = trials_df[
        trials_df["state"].astype(str).eq("COMPLETE")
    ].copy()

    save_parquet(completed_df, space["completed_trials_path"])

    optuna_trials_parts.append(trials_df)
    optuna_completed_parts.append(completed_df)

    print("Trials:", trials_df.shape)
    print("Completed:", completed_df.shape)
    print("Saved:", space["trials_path"])

optuna_trials_df = (
    pd.concat(optuna_trials_parts, ignore_index=True, sort=False)
    if optuna_trials_parts
    else pd.DataFrame()
)

optuna_completed_trials_df = (
    pd.concat(optuna_completed_parts, ignore_index=True, sort=False)
    if optuna_completed_parts
    else pd.DataFrame()
)

save_parquet(optuna_trials_df, OPTUNA_ALL_TRIALS_PATH)

print("All Optuna trials:", optuna_trials_df.shape)
print("All completed trials:", optuna_completed_trials_df.shape)

display(optuna_completed_trials_df.head(20))

c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-07-09 15:44:39,762] A new study created in RDB with name: simca_optuna_object_matrix_non_noisy_all


Optuna study for matrix_family=object_matrix


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-07-09 15:44:42,761] Trial 0 finished with values: [0.8113207547169812, 0.05454545454545454, 0.5670668953687822] and parameters: {'matrix_method': 'object_median', 'preprocessing': 'absorbance_snv_sg_d1', 'rule_variant': 'alternative_chi2_fixed2', 'n_components': 11, 'alpha': 0.01, 'sg_window_length': 15, 'sg_polyorder': 2, 'position_dilation_radius': 2}.
[I 2026-07-09 15:44:44,899] Trial 1 finished with values: [1.0, 0.0, 0.5] and parameters: {'matrix_method': 'object_mean', 'preprocessing': 'sg_smooth', 'rule_variant': 'alternative_chi2_emp_cv', 'n_components': 10, 'alpha': 0.05, 'sg_window_length': 11, 'sg_polyorder': 2, 'position_dilation_radius': 3}.
[I 2026-07-09 15:44:46,819] Trial 2 finished with values: [1.0, 0.0, 0.5] and parameters: {'matrix_method': 'object_mean', 'preprocessing': 'raw', 'rule_variant': 'alternative_chi2_emp_cv', 'n_components': 3, 'alpha': 0.01, 'position_dilation_radius': 4}.
[I 2026-07-09 15:44:49,527] Trial 3 finished with values: [0.452830188679

c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-07-09 16:03:54,116] A new study created in RDB with name: simca_optuna_pixel_matrix_non_noisy_all


  0%|          | 0/300 [00:00<?, ?it/s]

[I 2026-07-09 16:03:57,721] Trial 0 finished with values: [0.0, 0.6363636363636364, 0.681818181818182] and parameters: {'matrix_method': 'balanced_pixels', 'preprocessing': 'absorbance_sg_d2', 'rule_variant': 'alternative_empHQ_fixed2', 'n_components': 7, 'alpha': 0.05, 'm': 80, 'balanced_pixel_strategy': 'random', 'sg_window_length': 9, 'sg_polyorder': 2, 'position_dilation_radius': 2}.
[I 2026-07-09 16:04:01,143] Trial 1 finished with values: [0.0, 0.6181818181818182, 0.6909090909090909] and parameters: {'matrix_method': 'balanced_pixels', 'preprocessing': 'absorbance_sg_d2', 'rule_variant': 'simple_emp_cv', 'n_components': 7, 'alpha': 0.05, 'm': 20, 'balanced_pixel_strategy': 'random', 'sg_window_length': 9, 'sg_polyorder': 2, 'position_dilation_radius': 2}.
[I 2026-07-09 16:04:04,653] Trial 2 finished with values: [0.0, 1.0, 0.5] and parameters: {'matrix_method': 'balanced_pixels', 'preprocessing': 'snv_sg_smooth', 'rule_variant': 'alternative_chi2_emp_cv', 'n_components': 7, 'alph

,number,state,value_0,value_1,value_2,objective_fn_rate_max,objective_fp_rate_mean,objective_balanced_accuracy_mean,value,matrix_method,preprocessing,rule_variant,n_components,alpha,sg_window_length,sg_polyorder,position_dilation_radius,balanced_accuracy_mean,balanced_pixel_strategy,fn_rate_max,fn_rate_mean,fn_rate_std,fp_rate_max,fp_rate_mean,fp_rate_std,m,matrix_family,model_family,object_threshold_median,preprocessing_steps,selection_strategy,matrix_family_study
0,81,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,object_median,sg_smooth,data_driven_emp_cv,5,0.01,15,2,5,0.536364,random,0.000000,0.000000,0.0,0.927273,0.927273,0.000000e+00,40,object_matrix,rule_variant_grid,0.50,sg_smooth,optuna_binary_pareto,object_matrix
1,143,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,object_median,sg_smooth,data_driven_emp_cv,5,0.01,15,2,5,0.536364,random,0.000000,0.000000,0.0,0.927273,0.927273,0.000000e+00,40,object_matrix,rule_variant_grid,0.50,sg_smooth,optuna_binary_pareto,object_matrix
2,148,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,object_median,sg_smooth,alternative_empHQ_fixed2,5,0.01,15,2,3,0.536364,random,0.000000,0.000000,0.0,0.927273,0.927273,0.000000e+00,40,object_matrix,rule_variant_grid,0.50,sg_smooth,optuna_binary_pareto,object_matrix
3,149,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,object_median,sg_smooth,alternative_chi2_emp_cv,5,0.01,15,2,4,0.536364,random,0.000000,0.000000,0.0,0.927273,0.927273,0.000000e+00,40,object_matrix,rule_variant_grid,0.50,sg_smooth,optuna_binary_pareto,object_matrix
4,36,COMPLETE,0.000000,0.963636,0.518182,0.000000,0.963636,0.518182,NaN,object_median,absorbance_snv_sg_smooth,data_driven_emp_cv,4,0.01,15,2,5,0.518182,random,0.000000,0.000000,0.0,0.963636,0.963636,1.110223e-16,40,object_matrix,rule_variant_grid,0.55,absorbance+snv+sg_smooth,optuna_binary_pareto,object_matrix
5,64,COMPLETE,0.000000,0.963636,0.518182,0.000000,0.963636,0.518182,NaN,object_median,sg_smooth,simple_emp_cv,7,0.01,13,2,5,0.518182,random,0.000000,0.000000,0.0,0.963636,0.963636,1.110223e-16,40,object_matrix,rule_variant_grid,0.70,sg_smooth,optuna_binary_pareto,object_matrix
6,76,COMPLETE,0.000000,0.963636,0.518182,0.000000,0.963636,0.518182,NaN,object_median,snv_sg_smooth,alternative_chi2_emp_cv,4,0.01,15,2,5,0.518182,random,0.000000,0.000000,0.0,0.963636,0.963636,1.110223e-16,40,object_matrix,rule_variant_grid,0.55,snv+sg_smooth,optuna_binary_pareto,object_matrix
7,145,COMPLETE,0.000000,0.963636,0.518182,0.000000,0.963636,0.518182,NaN,object_median,snv_sg_smooth,alternative_empHQ_emp_cv,5,0.01,15,2,3,0.518182,random,0.000000,0.000000,0.0,0.963636,0.963636,1.110223e-16,40,object_matrix,rule_variant_grid,0.55,snv+sg_smooth,optuna_binary_pareto,object_matrix
8,91,COMPLETE,0.000000,0.981818,0.509091,0.000000,0.981818,0.509091,NaN,object_median,snv_sg_smooth,alternative_empHQ_fixed2,6,0.01,15,2,3,0.509091,random,0.000000,0.000000,0.0,0.981818,0.981818,0.000000e+00,40,object_matrix,rule_variant_grid,0.50,snv+sg_smooth,optuna_binary_pareto,object_matrix
9,108,COMPLETE,0.000000,0.981818,0.509091,0.000000,0.981818,0.509091,NaN,object_median,snv_sg_smooth,data_driven_emp_cv,3,0.01,7,2,2,0.509091,random,0.000000,0.000000,0.0,0.981818,0.981818,0.000000e+00,40,object_matrix,rule_variant_grid,0.55,snv+sg_smooth,optuna_binary_pareto,object_matrix


In [9]:
# ---------------------------------------------------------------------
# Harmonize objective columns for multi-objective Optuna studies
# ---------------------------------------------------------------------

objective_aliases = {
    "fn_rate_max": "value_0",
    "fp_rate_mean": "value_1",
    "balanced_accuracy_mean": "value_2",
}

for target_col, fallback_col in objective_aliases.items():
    if target_col not in optuna_completed_trials_df.columns and fallback_col in optuna_completed_trials_df.columns:
        optuna_completed_trials_df[target_col] = optuna_completed_trials_df[fallback_col]

for col in [
    "fn_rate_max",
    "fn_rate_mean",
    "fn_rate_std",
    "fp_rate_mean",
    "fp_rate_max",
    "fp_rate_std",
    "balanced_accuracy_mean",
    "object_threshold_median",
]:
    if col in optuna_completed_trials_df.columns:
        optuna_completed_trials_df[col] = pd.to_numeric(
            optuna_completed_trials_df[col],
            errors="coerce",
        )

In [13]:
# ---------------------------------------------------------------------
# Convert completed Optuna trials to Pareto candidate configs
# ---------------------------------------------------------------------

completed = optuna_completed_trials_df.copy()

if len(completed) == 0:
    optuna_pareto_candidates_df = pd.DataFrame()
else:
    # Harmonize objective columns from user_attrs.
    for col in [
        "fn_rate_max",
        "fn_rate_mean",
        "fn_rate_std",
        "fp_rate_mean",
        "fp_rate_max",
        "fp_rate_std",
        "balanced_accuracy_mean",
        "object_threshold_median",
    ]:
        if col in completed.columns:
            completed[col] = pd.to_numeric(completed[col], errors="coerce")

    if "matrix_family" not in completed.columns:
        completed["matrix_family"] = completed["matrix_family_study"]

    optuna_front_df = pareto_front_by_group(
        completed,
        group_cols=["matrix_family"],
        minimize_cols=[
            "fn_rate_max",
            "fp_rate_mean",
            "fn_rate_std",
        ],
        maximize_cols=[
            "balanced_accuracy_mean",
        ],
    )

    optuna_pareto_candidates_df = (
        optuna_front_df
        .sort_values(
            [
                "matrix_family",
                "fn_rate_max",
                "fp_rate_mean",
                "fn_rate_std",
                "balanced_accuracy_mean",
            ],
            ascending=[True, True, True, True, False],
        )
        .groupby("matrix_family", group_keys=False, dropna=False)
        .head(N_OPTUNA_PARETO_PER_FAMILY)
        .reset_index(drop=True)
    )

    optuna_pareto_candidates_df["optuna_trial_number"] = optuna_pareto_candidates_df["number"].astype(int)
    optuna_pareto_candidates_df["selected_config_id"] = [
        f"optuna_{row.matrix_family}_{int(row.optuna_trial_number):04d}"
        for row in optuna_pareto_candidates_df.itertuples()
    ]

    optuna_pareto_candidates_df["selection_split"] = "validation_batch_3"
    optuna_pareto_candidates_df["selection_strategy"] = "04B2_optuna_binary_pareto"
    optuna_pareto_candidates_df["candidate_source"] = "04B2_optuna_challenge"

    # The calibrated binary object threshold from the objective becomes the fixed threshold.
    optuna_pareto_candidates_df["object_threshold"] = optuna_pareto_candidates_df[
        "object_threshold_median"
    ].astype(float)

    # Harmonize rule columns.
    if "rule" not in optuna_pareto_candidates_df.columns and "rule_variant" in optuna_pareto_candidates_df.columns:
        optuna_pareto_candidates_df["rule"] = optuna_pareto_candidates_df["rule_variant"]

    optuna_pareto_candidates_df = ensure_candidate_columns(optuna_pareto_candidates_df)
    optuna_pareto_candidates_df = normalize_simca_rule_columns(optuna_pareto_candidates_df)
    optuna_pareto_candidates_df = fill_selected_config_defaults(
        optuna_pareto_candidates_df,
        default_values={
            "target_class": TARGET_CLASS,
            "non_target_label": NON_TARGET_LABEL,
            "sg_window_length": 11,
            "sg_polyorder": 2,
            "position_dilation_radius": 3,
            "m": 40,
            "alpha": 0.05,
            "object_threshold": 0.75,
        },
    )
    # Optuna trials contain aggregated multi-seed metrics.
    # We materialize canonical columns used by generic scoring helpers.
    OPTUNA_SELECTION_METRIC_ALIASES = {
        "fn_rate": [
            "fn_rate_max",
            "objective_fn_rate_max",
            "value_0",
            "fn_rate_mean",
        ],
        "fp_rate": [
            "fp_rate_mean",
            "objective_fp_rate_mean",
            "value_1",
            "fp_rate_max",
        ],
        "balanced_accuracy": [
            "balanced_accuracy_mean",
            "objective_balanced_accuracy_mean",
            "value_2",
        ],
    }

    optuna_pareto_candidates_df = materialize_selection_metrics(
        optuna_pareto_candidates_df,
        metric_aliases=OPTUNA_SELECTION_METRIC_ALIASES,
        overwrite=True,
        keep_source_columns=True,
    )

    optuna_pareto_candidates_df = add_detection_selection_score(
        optuna_pareto_candidates_df,
        metric_aliases=OPTUNA_SELECTION_METRIC_ALIASES,
        overwrite_metrics=False,
    )

    optuna_pareto_candidates_df = add_reference_selection_scores(
        optuna_pareto_candidates_df,
        metric_aliases=OPTUNA_SELECTION_METRIC_ALIASES,
        overwrite_metrics=False,
        score_prefix="optuna_",
    )

save_parquet(optuna_pareto_candidates_df, OPTUNA_PARETO_CANDIDATES_PATH)

print("Optuna Pareto candidates:", optuna_pareto_candidates_df.shape)
display(
    optuna_pareto_candidates_df[
        [
            col for col in [
                "selected_config_id",
                "matrix_family",
                "matrix_method",
                "preprocessing",
                "selected_rule_name",
                "rule_for_refit",
                "limit_source",
                "n_components",
                "alpha",
                "object_threshold",
                "m",
                "balanced_pixel_strategy",
                "fn_rate_max",
                "fp_rate_mean",
                "fn_rate_std",
                "balanced_accuracy_mean",
            ]
            if col in optuna_pareto_candidates_df.columns
        ]
    ]
)

Optuna Pareto candidates: (21, 67)


,selected_config_id,matrix_family,matrix_method,preprocessing,selected_rule_name,rule_for_refit,limit_source,n_components,alpha,object_threshold,m,balanced_pixel_strategy,fn_rate_max,fp_rate_mean,fn_rate_std,balanced_accuracy_mean
0,optuna_object_matrix_0081,object_matrix,object_median,sg_smooth,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,5,0.01,0.50,40,random,0.000000,0.927273,0.000000e+00,0.536364
1,optuna_object_matrix_0143,object_matrix,object_median,sg_smooth,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,5,0.01,0.50,40,random,0.000000,0.927273,0.000000e+00,0.536364
2,optuna_object_matrix_0148,object_matrix,object_median,sg_smooth,alternative_empHQ_fixed2,alternative_empHQ_fixed2,chi2,5,0.01,0.50,40,random,0.000000,0.927273,0.000000e+00,0.536364
3,optuna_object_matrix_0149,object_matrix,object_median,sg_smooth,alternative_chi2_emp_cv,alternative_chi2_emp_cv,empirical_cv,5,0.01,0.50,40,random,0.000000,0.927273,0.000000e+00,0.536364
4,optuna_object_matrix_0184,object_matrix,object_median,absorbance_sg_smooth,simple_emp_cv,simple_emp_cv,empirical_cv,8,0.01,0.50,40,random,0.018868,0.527273,0.000000e+00,0.726930
5,optuna_object_matrix_0185,object_matrix,object_median,absorbance_sg_smooth,simple_emp_cv,simple_emp_cv,empirical_cv,8,0.01,0.50,40,random,0.018868,0.527273,0.000000e+00,0.726930
6,optuna_object_matrix_0152,object_matrix,object_median,absorbance_sg_d2,alternative_empHQ_emp_cv,alternative_empHQ_emp_cv,empirical_cv,7,0.01,0.50,40,random,0.150943,0.418182,0.000000e+00,0.715437
7,optuna_object_matrix_0153,object_matrix,object_median,absorbance_sg_d2,alternative_empHQ_emp_cv,alternative_empHQ_emp_cv,empirical_cv,7,0.01,0.50,40,random,0.150943,0.418182,0.000000e+00,0.715437
8,optuna_object_matrix_0157,object_matrix,object_median,sg_smooth,combined_index_chi2,combined_index_chi2,scaled_chi2,3,0.01,0.50,40,random,0.207547,0.400000,2.775558e-17,0.696226
9,optuna_object_matrix_0160,object_matrix,object_median,sg_smooth,combined_index_chi2,combined_index_chi2,scaled_chi2,3,0.01,0.50,40,random,0.207547,0.400000,2.775558e-17,0.696226


In [15]:
# ---------------------------------------------------------------------
# Refit Optuna Pareto candidates on validation batch 3
# ---------------------------------------------------------------------

if len(optuna_pareto_candidates_df) == 0:
    optuna_validation_refit_metrics_df = pd.DataFrame()
    optuna_validation_refit_objects_df = pd.DataFrame()
    optuna_validation_refit_pixels_df = pd.DataFrame()
    optuna_validation_pixel_errors_by_image_df = pd.DataFrame()
    optuna_validation_refit_errors_df = pd.DataFrame()
    print("No Optuna candidate to refit.")

else:
    # Optional: limit refit per family to avoid too much runtime.
    optuna_refit_candidates_df = (
        optuna_pareto_candidates_df
        .sort_values(
            [
                "matrix_family",
                "fn_rate_max",
                "fp_rate_mean",
                "fn_rate_std",
                "balanced_accuracy_mean",
            ],
            ascending=[True, True, True, True, False],
        )
        .groupby("matrix_family", group_keys=False, dropna=False)
        .head(N_OPTUNA_REFIT_PER_FAMILY)
        .reset_index(drop=True)
    )

    (
        optuna_validation_refit_metrics_df,
        optuna_validation_refit_objects_df,
        optuna_validation_refit_pixels_df,
        optuna_validation_pixel_errors_by_image_df,
        optuna_validation_refit_errors_df,
    ) = refit_selected_simca_configs(
        selected_configs_df=optuna_refit_candidates_df,
        object_db=object_db,
        image_db=image_db,
        train_filters=TRAIN_FILTERS,
        projection_filters=VALIDATION_FILTERS,
        preprocessing_configs=PREPROCESSING_CONFIGS,
        evaluation_split="validation_batch_3_optuna_refit",
        wavelengths=wavelengths,
        random_state=RANDOM_STATE,
        replace=REPLACE_BALANCED_PIXELS,
        cv_n_splits=CV_N_SPLITS,
        cv_group_col=CV_GROUP_COL,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )

    optuna_validation_refit_objects_df = add_binary_confusion_case(
        optuna_validation_refit_objects_df,
        target_class=TARGET_CLASS,
        level="object",
    )

    optuna_validation_refit_pixels_df = add_binary_confusion_case(
        optuna_validation_refit_pixels_df,
        target_class=TARGET_CLASS,
        level="pixel",
    )

save_parquet(optuna_validation_refit_metrics_df, OPTUNA_VALIDATION_REFIT_METRICS_PATH)
save_parquet_if_nonempty(optuna_validation_refit_objects_df, OPTUNA_VALIDATION_REFIT_OBJECTS_PATH)
save_parquet_if_nonempty(optuna_validation_refit_pixels_df, OPTUNA_VALIDATION_REFIT_PIXELS_PATH)
save_parquet_if_nonempty(optuna_validation_pixel_errors_by_image_df, OPTUNA_VALIDATION_PIXEL_ERRORS_BY_IMAGE_PATH)
save_parquet_if_nonempty(optuna_validation_refit_errors_df, OPTUNA_VALIDATION_REFIT_ERRORS_PATH)

print("Optuna validation refit metrics:", optuna_validation_refit_metrics_df.shape)
print("Optuna validation refit errors:", optuna_validation_refit_errors_df.shape)

display(optuna_validation_refit_metrics_df.head())
display(optuna_validation_refit_errors_df)

[validation_batch_3_optuna_refit] optuna_object_matrix_0081
[validation_batch_3_optuna_refit] optuna_object_matrix_0143
[validation_batch_3_optuna_refit] optuna_object_matrix_0148
[validation_batch_3_optuna_refit] optuna_object_matrix_0149
[validation_batch_3_optuna_refit] optuna_object_matrix_0184
[validation_batch_3_optuna_refit] optuna_object_matrix_0185
[validation_batch_3_optuna_refit] optuna_object_matrix_0152
[validation_batch_3_optuna_refit] optuna_object_matrix_0153
[validation_batch_3_optuna_refit] optuna_object_matrix_0157
[validation_batch_3_optuna_refit] optuna_object_matrix_0160
[validation_batch_3_optuna_refit] optuna_object_matrix_0105
[validation_batch_3_optuna_refit] optuna_object_matrix_0106
[validation_batch_3_optuna_refit] optuna_object_matrix_0121
[validation_batch_3_optuna_refit] optuna_pixel_matrix_0111
[validation_batch_3_optuna_refit] optuna_pixel_matrix_0112
[validation_batch_3_optuna_refit] optuna_pixel_matrix_0285
[validation_batch_3_optuna_refit] optuna_pi

,number,state,value_0,value_1,value_2,objective_fn_rate_max,objective_fp_rate_mean,objective_balanced_accuracy_mean,value,matrix_method,preprocessing,rule_variant,n_components,alpha,sg_window_length,sg_polyorder,position_dilation_radius,balanced_accuracy_mean,balanced_pixel_strategy,fn_rate_max,fn_rate_mean,fn_rate_std,fp_rate_max,fp_rate_mean,fp_rate_std,m,matrix_family,model_family,object_threshold_median,preprocessing_steps,selection_strategy,matrix_family_study,optuna_trial_number,selected_config_id,selection_split,candidate_source,object_threshold,rule,training_matrix_id,selected_rule_name,rule_for_refit,rule_original,rule_variant_original,rule_token,limit_source,target_class,non_target_label,fn_rate,fn_rate_source,fp_rate,fp_rate_source,balanced_accuracy,balanced_accuracy_source,target_sensitivity,target_sensitivity_source,non_target_specificity,non_target_specificity_source,f1_score,f1_score_source,accuracy,accuracy_source,precision,precision_source,selection_score,optuna_score_conservative_target,optuna_score_balanced_reference,optuna_score_specificity_control,non_target_class,n,tp,fn,fp,tn,evaluation_split,n_projected_objects,n_projected_pixels
0,81,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,object_median,sg_smooth,data_driven_emp_cv,5,0.01,15,2,5,0.536364,random,0.000000,0.000000,0.0,0.927273,0.927273,0.0,40,object_matrix,rule_variant_grid,0.5,sg_smooth,04B2_optuna_binary_pareto,object_matrix,81,optuna_object_matrix_0081,validation_batch_3,04B2_optuna_challenge,0.5,data_driven,object_median,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,peanut,non_target,0.000000,fn_rate_max,0.927273,fp_rate_mean,0.536364,balanced_accuracy_mean,1.000000,None,0.072727,None,0.675159,None,0.527778,None,0.509615,None,-0.927273,-1.318182,0.681818,-2.563636,non_target,108,53,0,51,4,validation_batch_3_optuna_refit,108,6812
1,143,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,object_median,sg_smooth,data_driven_emp_cv,5,0.01,15,2,5,0.536364,random,0.000000,0.000000,0.0,0.927273,0.927273,0.0,40,object_matrix,rule_variant_grid,0.5,sg_smooth,04B2_optuna_binary_pareto,object_matrix,143,optuna_object_matrix_0143,validation_batch_3,04B2_optuna_challenge,0.5,data_driven,object_median,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,peanut,non_target,0.000000,fn_rate_max,0.927273,fp_rate_mean,0.536364,balanced_accuracy_mean,1.000000,None,0.072727,None,0.675159,None,0.527778,None,0.509615,None,-0.927273,-1.318182,0.681818,-2.563636,non_target,108,53,0,51,4,validation_batch_3_optuna_refit,108,6812
2,148,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,object_median,sg_smooth,alternative_empHQ_fixed2,5,0.01,15,2,3,0.536364,random,0.000000,0.000000,0.0,0.927273,0.927273,0.0,40,object_matrix,rule_variant_grid,0.5,sg_smooth,04B2_optuna_binary_pareto,object_matrix,148,optuna_object_matrix_0148,validation_batch_3,04B2_optuna_challenge,0.5,alternative,object_median,alternative_empHQ_fixed2,alternative_empHQ_fixed2,alternative_empHQ_fixed2,alternative_empHQ_fixed2,alternative_empHQ_fixed2,chi2,peanut,non_target,0.000000,fn_rate_max,0.927273,fp_rate_mean,0.536364,balanced_accuracy_mean,1.000000,None,0.072727,None,0.675159,None,0.527778,None,0.509615,None,-0.927273,-1.318182,0.681818,-2.563636,non_target,108,53,0,51,4,validation_batch_3_optuna_refit,108,6812
3,149,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,object_median,sg_smooth,alternative_chi2_emp_cv,5,0.01,15,2,4,0.536364,random,0.000000,0.000000,0.0,0.927273,0.927273,0.0,40,object_matrix,rule_variant_grid,0.5,sg_smooth,04B2_optuna_binary_pareto,object_matrix,149,optuna_object_matrix_0149,validation_batch_3,04B2_optuna_challenge,0.5,alternative,object_median,alternative_chi2_emp_cv,alternative_chi2_emp_cv,alternative_chi2_emp_cv,alternative_chi2_emp_cv,alternative_chi2_emp_cv,empirical_cv,peanut,non_targe

""


In [16]:
# ---------------------------------------------------------------------
# Calibrate 3-way thresholds for Optuna challengers on validation batch 3
# ---------------------------------------------------------------------

if len(optuna_validation_refit_objects_df) == 0:
    optuna_three_way_grid_df = pd.DataFrame()
    optuna_three_way_selected_thresholds_df = pd.DataFrame()
    optuna_validation_3way_metrics_df = pd.DataFrame()
    optuna_validation_3way_objects_df = pd.DataFrame()
    optuna_challengers_df = pd.DataFrame()

else:
    optuna_three_way_grid_df, optuna_three_way_selected_thresholds_df = calibrate_three_way_thresholds_by_config(
        object_df=optuna_validation_refit_objects_df,
        config_cols=["selected_config_id"],
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
        lower_thresholds=THREE_WAY_LOWER_THRESHOLDS,
        upper_thresholds=THREE_WAY_UPPER_THRESHOLDS,
        max_target_miss_rate=MAX_THREE_WAY_TARGET_MISS_RATE,
        max_false_accept_rate=MAX_THREE_WAY_FALSE_ACCEPT_RATE,
        max_uncertain_rate=MAX_THREE_WAY_UNCERTAIN_RATE,
    )

    optuna_validation_3way_metrics_df, optuna_validation_3way_objects_df = evaluate_three_way_by_config(
        object_df=optuna_validation_refit_objects_df,
        thresholds_df=optuna_three_way_selected_thresholds_df,
        config_id_col="selected_config_id",
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )

    optuna_challengers_df = optuna_pareto_candidates_df.merge(
        optuna_three_way_selected_thresholds_df[
            [
                "selected_config_id",
                "three_way_lower_threshold",
                "three_way_upper_threshold",
                "target_miss_rate",
                "screening_sensitivity",
                "non_target_false_accept_rate",
                "uncertain_rate",
                "coverage_rate",
            ]
        ].rename(
            columns={
                "target_miss_rate": "validation_3way_target_miss_rate",
                "screening_sensitivity": "validation_3way_screening_sensitivity",
                "non_target_false_accept_rate": "validation_3way_non_target_false_accept_rate",
                "uncertain_rate": "validation_3way_uncertain_rate",
                "coverage_rate": "validation_3way_coverage_rate",
            }
        ),
        on="selected_config_id",
        how="inner",
    )

    # Add refit binary metrics from validation.
    refit_cols = [
        "selected_config_id",
        "balanced_accuracy",
        "target_sensitivity",
        "non_target_specificity",
        "fn_rate",
        "fp_rate",
        "f1_score",
        "accuracy",
        "precision",
    ]
    refit_cols = [
        col for col in refit_cols
        if col in optuna_validation_refit_metrics_df.columns
    ]

    optuna_challengers_df = optuna_challengers_df.merge(
        optuna_validation_refit_metrics_df[refit_cols].rename(
            columns={
                "balanced_accuracy": "validation_balanced_accuracy",
                "target_sensitivity": "validation_target_sensitivity",
                "non_target_specificity": "validation_non_target_specificity",
                "fn_rate": "validation_fn_rate",
                "fp_rate": "validation_fp_rate",
                "f1_score": "validation_f1_score",
                "accuracy": "validation_accuracy",
                "precision": "validation_precision",
            }
        ),
        on="selected_config_id",
        how="left",
    )

    optuna_challengers_df["candidate_source"] = "04B2_optuna_challenge"
    optuna_challengers_df["selection_strategy"] = (
        optuna_challengers_df["selection_strategy"].astype(str)
        + "__3way_calibrated"
    )

save_parquet_if_nonempty(optuna_three_way_grid_df, OPTUNA_THREE_WAY_GRID_PATH)
save_parquet_if_nonempty(optuna_three_way_selected_thresholds_df, OPTUNA_THREE_WAY_SELECTED_PATH)
save_parquet_if_nonempty(optuna_validation_3way_objects_df, OPTUNA_VALIDATION_3WAY_OBJECTS_PATH)
save_parquet_if_nonempty(optuna_validation_3way_metrics_df, OPTUNA_VALIDATION_3WAY_METRICS_PATH)
save_parquet_if_nonempty(optuna_challengers_df, OPTUNA_CHALLENGERS_FOR_04C_PATH)

print("Optuna 3-way grid:", optuna_three_way_grid_df.shape)
print("Optuna selected 3-way thresholds:", optuna_three_way_selected_thresholds_df.shape)
print("Optuna challengers for 04C:", optuna_challengers_df.shape)

display(
    optuna_challengers_df[
        [
            col for col in [
                "selected_config_id",
                "matrix_family",
                "matrix_method",
                "preprocessing",
                "selected_rule_name",
                "n_components",
                "alpha",
                "object_threshold",
                "three_way_lower_threshold",
                "three_way_upper_threshold",
                "validation_fn_rate",
                "validation_fp_rate",
                "validation_3way_target_miss_rate",
                "validation_3way_non_target_false_accept_rate",
                "validation_3way_uncertain_rate",
                "validation_3way_coverage_rate",
            ]
            if col in optuna_challengers_df.columns
        ]
    ]
)

Optuna 3-way grid: (2394, 23)
Optuna selected 3-way thresholds: (21, 23)
Optuna challengers for 04C: (21, 82)


,selected_config_id,matrix_family,matrix_method,preprocessing,selected_rule_name,n_components,alpha,object_threshold,three_way_lower_threshold,three_way_upper_threshold,validation_fn_rate,validation_fp_rate,validation_3way_target_miss_rate,validation_3way_non_target_false_accept_rate,validation_3way_uncertain_rate,validation_3way_coverage_rate
0,optuna_object_matrix_0081,object_matrix,object_median,sg_smooth,data_driven_emp_cv,5,0.01,0.50,0.50,0.95,0.000000,0.927273,0.000000,0.290909,0.546296,0.453704
1,optuna_object_matrix_0143,object_matrix,object_median,sg_smooth,data_driven_emp_cv,5,0.01,0.50,0.50,0.95,0.000000,0.927273,0.000000,0.290909,0.546296,0.453704
2,optuna_object_matrix_0148,object_matrix,object_median,sg_smooth,alternative_empHQ_fixed2,5,0.01,0.50,0.50,0.95,0.000000,0.927273,0.000000,0.236364,0.601852,0.398148
3,optuna_object_matrix_0149,object_matrix,object_median,sg_smooth,alternative_chi2_emp_cv,5,0.01,0.50,0.50,0.95,0.000000,0.927273,0.000000,0.345455,0.509259,0.490741
4,optuna_object_matrix_0184,object_matrix,object_median,absorbance_sg_smooth,simple_emp_cv,8,0.01,0.50,0.30,0.70,0.018868,0.527273,0.000000,0.290909,0.564815,0.435185
5,optuna_object_matrix_0185,object_matrix,object_median,absorbance_sg_smooth,simple_emp_cv,8,0.01,0.50,0.30,0.70,0.018868,0.527273,0.000000,0.290909,0.564815,0.435185
6,optuna_object_matrix_0152,object_matrix,object_median,absorbance_sg_d2,alternative_empHQ_emp_cv,7,0.01,0.50,0.25,0.65,0.150943,0.418182,0.000000,0.290909,0.500000,0.500000
7,optuna_object_matrix_0153,object_matrix,object_median,absorbance_sg_d2,alternative_empHQ_emp_cv,7,0.01,0.50,0.25,0.65,0.150943,0.418182,0.000000,0.290909,0.500000,0.500000
8,optuna_object_matrix_0157,object_matrix,object_median,sg_smooth,combined_index_chi2,3,0.01,0.50,0.05,0.65,0.207547,0.400000,0.000000,0.272727,0.555556,0.444444
9,optuna_object_matrix_0160,object_matrix,object_median,sg_smooth,combined_index_chi2,3,0.01,0.50,0.05,0.65,0.207547,0.400000,0.000000,0.272727,0.555556,0.444444


In [17]:
# ---------------------------------------------------------------------
# Merge 04B robust panel and Optuna challengers for 04C
# ---------------------------------------------------------------------

candidate_parts = []

if INCLUDE_GRID_ROBUST_CANDIDATES and len(robust_grid_candidates_df) > 0:
    grid_for_test_df = robust_grid_candidates_df.copy()
    grid_for_test_df["candidate_source"] = grid_for_test_df.get(
        "candidate_source",
        "04B_grid_robustness",
    )
    candidate_parts.append(grid_for_test_df)

if len(optuna_challengers_df) > 0:
    optuna_for_test_df = optuna_challengers_df.copy()
    optuna_for_test_df["candidate_source"] = "04B2_optuna_challenge"
    candidate_parts.append(optuna_for_test_df)

if candidate_parts:
    candidate_configs_for_pure_test_df = (
        pd.concat(candidate_parts, ignore_index=True, sort=False)
        .drop_duplicates(
            subset=[
                "model_family",
                "matrix_family",
                "training_matrix_id",
                "matrix_method",
                "preprocessing",
                "selected_rule_name",
                "rule_for_refit",
                "n_components",
                "alpha",
                "object_threshold",
                "m",
                "balanced_pixel_strategy",
                "sg_window_length",
                "sg_polyorder",
                "position_dilation_radius",
                "three_way_lower_threshold",
                "three_way_upper_threshold",
            ],
            keep="first",
        )
        .reset_index(drop=True)
    )
else:
    candidate_configs_for_pure_test_df = pd.DataFrame()

candidate_configs_for_pure_test_df = ensure_candidate_columns(candidate_configs_for_pure_test_df)
candidate_configs_for_pure_test_df = normalize_simca_rule_columns(candidate_configs_for_pure_test_df)
candidate_configs_for_pure_test_df = fill_selected_config_defaults(
    candidate_configs_for_pure_test_df,
    default_values={
        "target_class": TARGET_CLASS,
        "non_target_label": NON_TARGET_LABEL,
        "sg_window_length": 11,
        "sg_polyorder": 2,
        "position_dilation_radius": 3,
        "m": 40,
        "alpha": 0.05,
        "object_threshold": 0.75,
    },
)

# Keep families separated in the final display/order.
sort_cols = [
    "matrix_family",
    "candidate_source",
    "validation_fn_rate",
    "validation_fp_rate",
    "validation_3way_target_miss_rate",
    "validation_3way_uncertain_rate",
]

sort_cols = [
    col for col in sort_cols
    if col in candidate_configs_for_pure_test_df.columns
]

ascending = [
    True if col in {"matrix_family", "candidate_source"} else True
    for col in sort_cols
]

candidate_configs_for_pure_test_df = (
    candidate_configs_for_pure_test_df
    .sort_values(sort_cols, ascending=ascending)
    .reset_index(drop=True)
)

# Ensure unique selected_config_id.
if candidate_configs_for_pure_test_df["selected_config_id"].astype(str).duplicated().any():
    candidate_configs_for_pure_test_df["previous_selected_config_id"] = (
        candidate_configs_for_pure_test_df["selected_config_id"].astype(str)
    )
    candidate_configs_for_pure_test_df["selected_config_id"] = [
        f"testcand_{i:03d}" for i in range(len(candidate_configs_for_pure_test_df))
    ]

required_for_04c = [
    "selected_config_id",
    "matrix_family",
    "matrix_method",
    "preprocessing",
    "rule_for_refit",
    "n_components",
    "alpha",
    "object_threshold",
    "three_way_lower_threshold",
    "three_way_upper_threshold",
]

missing_for_04c = [
    col for col in required_for_04c
    if col not in candidate_configs_for_pure_test_df.columns
]

if missing_for_04c:
    raise KeyError(f"Missing required columns for 04C: {missing_for_04c}")

save_parquet(
    candidate_configs_for_pure_test_df,
    CANDIDATE_CONFIGS_FOR_PURE_TEST_PATH,
)

print("Candidate configs for pure test:", candidate_configs_for_pure_test_df.shape)
print("Saved:", CANDIDATE_CONFIGS_FOR_PURE_TEST_PATH)

display(candidate_configs_for_pure_test_df["matrix_family"].value_counts(dropna=False))
display(candidate_configs_for_pure_test_df["candidate_source"].value_counts(dropna=False))

display(
    candidate_configs_for_pure_test_df[
        [
            col for col in [
                "selected_config_id",
                "candidate_source",
                "selection_strategy",
                "matrix_family",
                "training_matrix_id",
                "model_family",
                "matrix_method",
                "preprocessing",
                "selected_rule_name",
                "n_components",
                "alpha",
                "object_threshold",
                "three_way_lower_threshold",
                "three_way_upper_threshold",
                "validation_fn_rate",
                "validation_fp_rate",
                "validation_3way_target_miss_rate",
                "validation_3way_uncertain_rate",
            ]
            if col in candidate_configs_for_pure_test_df.columns
        ]
    ].head(80)
)

Candidate configs for pure test: (31, 101)
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B2_optuna_challenge_non_noisy_all\candidate_configs_for_pure_test.parquet


matrix_family
object_matrix    20
pixel_matrix     11
Name: count, dtype: int64

candidate_source
04B_grid_robustness      18
04B2_optuna_challenge    13
Name: count, dtype: int64

,selected_config_id,candidate_source,selection_strategy,matrix_family,training_matrix_id,model_family,matrix_method,preprocessing,selected_rule_name,n_components,alpha,object_threshold,three_way_lower_threshold,three_way_upper_threshold,validation_fn_rate,validation_fp_rate,validation_3way_target_miss_rate,validation_3way_uncertain_rate
0,optuna_object_matrix_0149,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_chi2_emp_cv,5,0.01,0.50,0.50,0.95,0.000000,0.927273,0.000000,0.509259
1,optuna_object_matrix_0081,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,data_driven_emp_cv,5,0.01,0.50,0.50,0.95,0.000000,0.927273,0.000000,0.546296
2,optuna_object_matrix_0148,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_empHQ_fixed2,5,0.01,0.50,0.50,0.95,0.000000,0.927273,0.000000,0.601852
3,optuna_object_matrix_0184,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_median,rule_variant_grid,object_median,absorbance_sg_smooth,simple_emp_cv,8,0.01,0.50,0.30,0.70,0.018868,0.527273,0.000000,0.564815
4,optuna_object_matrix_0152,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_median,rule_variant_grid,object_median,absorbance_sg_d2,alternative_empHQ_emp_cv,7,0.01,0.50,0.25,0.65,0.150943,0.418182,0.000000,0.500000
5,optuna_object_matrix_0157,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,combined_index_chi2,3,0.01,0.50,0.05,0.65,0.207547,0.400000,0.000000,0.555556
6,optuna_object_matrix_0105,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_mean,rule_variant_grid,object_mean,snv_sg_smooth,alternative_empHQ_fixed2,3,0.01,0.55,0.15,0.55,0.264151,0.000000,0.000000,0.416667
7,optuna_object_matrix_0121,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_mean,rule_variant_grid,object_mean,snv_sg_smooth,alternative_empHQ_fixed2,3,0.01,0.55,0.15,0.55,0.264151,0.000000,0.000000,0.416667
8,04A_object_matrix_0001,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_mean,empirical_cv_rule,object_mean,absorbance_sg_smooth,data_driven_emp_cv,<NA>,0.01,0.75,0.05,0.95,0.000000,0.945455,0.000000,0.435185
9,04A_object_matrix_0002,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_median,empirical_cv_rule,object_median,absorbance_sg_d2,data_driven_emp_cv,<NA>,0.01,0.70,0.55,0.95,0.018868,0.781818,0.000000,0.712963


In [18]:
optuna_challenge_protocol_df = pd.DataFrame([{
    "db_h5_path": str(DB_H5_PATH),
    "results_dir": str(RESULTS_DIR),

    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_active_bands": int(len(wavelengths)),

    "target_class": TARGET_CLASS,
    "non_target_label": NON_TARGET_LABEL,
    "reference_classes_json": json.dumps(list(REFERENCE_CLASSES)),

    "train_filters_json": json.dumps(TRAIN_FILTERS, default=str),
    "validation_filters_json": json.dumps(VALIDATION_FILTERS, default=str),

    "pca_selected_preprocessings_path": str(PCA_SELECTED_PREPROCESSINGS_PATH),
    "grid_selected_candidates_path": str(GRID_SELECTED_CANDIDATES_PATH),
    "robust_candidate_configs_path": str(ROBUST_CANDIDATE_CONFIGS_PATH),

    "optuna_storage_object_path": str(OPTUNA_STORAGE_OBJECT_PATH),
    "optuna_storage_pixel_path": str(OPTUNA_STORAGE_PIXEL_PATH),
    "run_optuna": bool(RUN_OPTUNA),
    "load_existing_study": bool(LOAD_EXISTING_STUDY),
    "n_trials_object_matrix": int(OPTUNA_N_TRIALS_PER_FAMILY["object_matrix"]),
    "n_trials_pixel_matrix": int(OPTUNA_N_TRIALS_PER_FAMILY["pixel_matrix"]),
    "timeout": OPTUNA_TIMEOUT,
    "n_jobs": int(OPTUNA_N_JOBS),
    "random_state": int(RANDOM_STATE),

    "matrix_family_spaces_json": json.dumps(
        {
            fam: {
                key: str(value)
                for key, value in space.items()
                if key != "storage_path"
            }
            for fam, space in OPTUNA_MATRIX_FAMILY_SPACES.items()
        },
        default=str,
    ),

    "rule_variants_json": json.dumps(OPTUNA_RULE_VARIANTS),
    "n_components_choices_json": json.dumps(OPTUNA_N_COMPONENTS_CHOICES),
    "alpha_choices_json": json.dumps(OPTUNA_ALPHA_CHOICES),
    "object_thresholds_json": json.dumps([float(v) for v in OPTUNA_OBJECT_THRESHOLDS]),
    "objective_seeds_json": json.dumps(OPTUNA_OBJECTIVE_SEEDS),

    "m_choices_json": json.dumps(OPTUNA_M_CHOICES),
    "balanced_pixel_strategy_choices_json": json.dumps(OPTUNA_BALANCED_PIXEL_STRATEGY_CHOICES),
    "sg_window_choices_json": json.dumps(OPTUNA_SG_WINDOW_CHOICES),
    "sg_polyorder_choices_json": json.dumps(OPTUNA_SG_POLYORDER_CHOICES),
    "position_dilation_radius_choices_json": json.dumps(OPTUNA_POSITION_DILATION_RADIUS_CHOICES),

    "preprocessing_configs_json": json.dumps(
        {name: list(steps) for name, steps in PREPROCESSING_CONFIGS.items()},
        default=str,
    ),

    "n_trials_total": int(len(optuna_trials_df)),
    "n_trials_completed": int(len(optuna_completed_trials_df)),
    "n_optuna_pareto_candidates": int(len(optuna_pareto_candidates_df)),
    "n_optuna_challengers": int(len(optuna_challengers_df)),
    "n_candidate_configs_for_pure_test": int(len(candidate_configs_for_pure_test_df)),

    "optuna_trials_all_path": str(OPTUNA_ALL_TRIALS_PATH),
    "optuna_trials_object_path": str(OPTUNA_TRIALS_OBJECT_PATH),
    "optuna_trials_pixel_path": str(OPTUNA_TRIALS_PIXEL_PATH),
    "optuna_pareto_candidates_path": str(OPTUNA_PARETO_CANDIDATES_PATH),
    "optuna_validation_refit_metrics_path": str(OPTUNA_VALIDATION_REFIT_METRICS_PATH),
    "optuna_validation_3way_metrics_path": str(OPTUNA_VALIDATION_3WAY_METRICS_PATH),
    "optuna_challengers_for_04C_path": str(OPTUNA_CHALLENGERS_FOR_04C_PATH),
    "candidate_configs_for_pure_test_path": str(CANDIDATE_CONFIGS_FOR_PURE_TEST_PATH),
}])

save_parquet(
    optuna_challenge_protocol_df,
    OPTUNA_CHALLENGE_PROTOCOL_PATH,
)

print("Saved Optuna challenge protocol:")
print(OPTUNA_CHALLENGE_PROTOCOL_PATH)

display(optuna_challenge_protocol_df)

Saved Optuna challenge protocol:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B2_optuna_challenge_non_noisy_all\optuna_challenge_protocol.parquet


,db_h5_path,results_dir,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_active_bands,target_class,non_target_label,reference_classes_json,train_filters_json,validation_filters_json,pca_selected_preprocessings_path,grid_selected_candidates_path,robust_candidate_configs_path,optuna_storage_object_path,optuna_storage_pixel_path,run_optuna,load_existing_study,n_trials_object_matrix,n_trials_pixel_matrix,timeout,n_jobs,random_state,matrix_family_spaces_json,rule_variants_json,n_components_choices_json,alpha_choices_json,object_thresholds_json,objective_seeds_json,m_choices_json,balanced_pixel_strategy_choices_json,sg_window_choices_json,sg_polyorder_choices_json,position_dilation_radius_choices_json,preprocessing_configs_json,n_trials_total,n_trials_completed,n_optuna_pareto_candidates,n_optuna_challengers,n_candidate_configs_for_pure_test,optuna_trials_all_path,optuna_trials_object_path,optuna_trials_pixel_path,optuna_pareto_candidates_path,optuna_validation_refit_metrics_path,optuna_validation_3way_metrics_path,optuna_challengers_for_04C_path,candidate_configs_for_pure_test_path
0,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,non_noisy_all,False,non_noisy_all,NaN,NaN,63,peanut,non_target,"[""almond"", ""peanut""]","{""sample_kind"": [""pure""], ""object_nut_type"": [...","{""sample_kind"": [""pure""], ""object_nut_type"": [...",C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,True,False,200,300,None,1,42,"{""object_matrix"": {""matrix_methods"": ""['object...","[""simple_chi2"", ""simple_emp_cv"", ""alternative_...","[3, 4, 5, 6, 7, 8, 10, 11, 12]","[0.05, 0.01]","[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0...","[0, 1, 2]","[20, 40, 60, 80]","[""random"", ""center""]","[7, 9, 11, 13, 15]",[2],"[2, 3, 4, 5]","{""absorbance_sg_smooth"": [""absorbance"", ""sg_sm...",500,500,21,21,31,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...


In [19]:
print("04B2_optuna_challenge_optional.ipynb completed.")
print()
print("Essential outputs:")
print(" -", OPTUNA_TRIALS_OBJECT_PATH)
print(" -", OPTUNA_TRIALS_PIXEL_PATH)
print(" -", OPTUNA_ALL_TRIALS_PATH)
print(" -", OPTUNA_PARETO_CANDIDATES_PATH)
print(" -", OPTUNA_VALIDATION_REFIT_METRICS_PATH)
print(" -", OPTUNA_VALIDATION_3WAY_METRICS_PATH)
print(" -", OPTUNA_CHALLENGERS_FOR_04C_PATH)
print(" -", CANDIDATE_CONFIGS_FOR_PURE_TEST_PATH)
print(" -", OPTUNA_CHALLENGE_PROTOCOL_PATH)

print()
print("Summary:")
print(f" - Total Optuna trials: {len(optuna_trials_df)}")
print(f" - Completed Optuna trials: {len(optuna_completed_trials_df)}")
print(f" - Optuna Pareto candidates: {len(optuna_pareto_candidates_df)}")
print(f" - Optuna challengers for 04C: {len(optuna_challengers_df)}")
print(f" - Candidate configs for pure test: {len(candidate_configs_for_pure_test_df)}")
print()
print("Next notebook:")
print("04C_simca_pure_test_evaluation.ipynb")

04B2_optuna_challenge_optional.ipynb completed.

Essential outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B2_optuna_challenge_non_noisy_all\optuna_trials_object_matrix.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B2_optuna_challenge_non_noisy_all\optuna_trials_pixel_matrix.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B2_optuna_challenge_non_noisy_all\optuna_trials_all_families.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B2_optuna_challenge_non_noisy_all\optuna_pareto_candidates.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B2_optuna_challenge_non_noisy_all\optuna_validation_refit_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B2_optuna_challenge_non_noisy_all\optuna_validation_3way_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\re